In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [10]:
import os
os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

# Load dysregulation scores (478 patients × 819 genes)
dysreg = pd.read_csv('data/processed/dysregulation_scores.csv', index_col=0)

# Load clinical survival labels
clinical = pd.read_csv('data/processed/clinical_survival.csv', index_col=0)

# Align patients — keep only patients present in both
common_patients = dysreg.index.intersection(clinical.index)
dysreg = dysreg.loc[common_patients]
clinical = clinical.loc[common_patients]

print(f"Dysregulation matrix: {dysreg.shape}")
print(f"Clinical data:        {clinical.shape}")
print(f"Patients aligned:     {len(common_patients)}")
print(f"\nSurvival columns: {list(clinical.columns)}")

Dysregulation matrix: (478, 819)
Clinical data:        (478, 10)
Patients aligned:     478

Survival columns: ['vital_status', 'days_to_death', 'days_to_last_followup', 'age', 'gender', 'stage', 'survival_time', 'event', 'stage_group', 'age_group']


In [11]:
# Build structured array for sksurv
# event = 1 means patient died, 0 means censored (still alive at last follow-up)
# survival_time = days survived

y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"Total patients:  {len(y)}")
print(f"Events (deaths): {y['event'].sum()}  ({y['event'].mean()*100:.1f}%)")
print(f"Censored:        {(~y['event']).sum()}  ({(~y['event']).mean()*100:.1f}%)")
print(f"Survival time range: {y['time'].min():.0f} to {y['time'].max():.0f} days")

Total patients:  478
Events (deaths): 121  (25.3%)
Censored:        357  (74.7%)
Survival time range: 1 to 6812 days


In [12]:
# Standardise dysregulation scores
# z-scores from Sonica already have meaning (deviation from healthy)
# but the scale varies wildly across genes (max was 80)
# StandardScaler brings everything to mean=0, std=1
# so no single gene dominates just because it has a large raw z-score

scaler = StandardScaler()
X = scaler.fit_transform(dysreg)
X = pd.DataFrame(X, index=dysreg.index, columns=dysreg.columns)

print(f"Feature matrix shape: {X.shape}")
print(f"After scaling:")
print(f"  Mean (should be ~0): {X.values.mean():.6f}")
print(f"  Std  (should be ~1): {X.values.std():.6f}")
print(f"  Min: {X.values.min():.3f}")
print(f"  Max: {X.values.max():.3f}")

Feature matrix shape: (478, 819)
After scaling:
  Mean (should be ~0): 0.000000
  Std  (should be ~1): 1.000000
  Min: -7.005
  Max: 8.533


In [13]:
from sklearn.model_selection import KFold

# Start alpha from higher value — 819 features needs stronger regularisation
alphas = np.logspace(-1, 2, 50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex = []
fold_alphas = []
fold_genes  = []

print("Running 5-fold CV Cox-Lasso on dysregulation features...")
print(f"{'Fold':<6} {'Best Alpha':<12} {'Genes kept':<12} {'C-index':<10}")
print("-" * 44)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    cox = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=alphas,
                                  fit_baseline_model=True)
    cox.fit(X_train, y_train)

    best_alpha = cox.alphas_[0]
    best_cindex = 0
    for a in cox.alphas_:
        cox_a = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[a],
                                        fit_baseline_model=True)
        cox_a.fit(X_train, y_train)
        ci = concordance_index_censored(y_train['event'],
                                         y_train['time'],
                                         cox_a.predict(X_train))[0]
        if ci > best_cindex:
            best_cindex = ci
            best_alpha = a

    cox_best = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[best_alpha],
                                       fit_baseline_model=True)
    cox_best.fit(X_train, y_train)

    risk_scores = cox_best.predict(X_test)
    ci_test = concordance_index_censored(y_test['event'],
                                          y_test['time'],
                                          risk_scores)[0]

    n_genes = np.sum(cox_best.coef_[:, 0] != 0)
    fold_cindex.append(ci_test)
    fold_alphas.append(best_alpha)
    fold_genes.append(n_genes)

    print(f"{fold:<6} {best_alpha:<12.4f} {n_genes:<12} {ci_test:.4f}")

print("-" * 44)
print(f"\nMean C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")

Running 5-fold CV Cox-Lasso on dysregulation features...
Fold   Best Alpha   Genes kept   C-index   
--------------------------------------------
1      0.1000       2            0.5346
2      0.1000       2            0.6723
3      0.1000       7            0.5238
4      0.1000       1            0.5455
5      0.1000       3            0.5201
--------------------------------------------

Mean C-index: 0.559 ± 0.057


In [14]:
import json
import pickle

# Save final model fitted on ALL data
cox_final = CoxnetSurvivalAnalysis(l1_ratio=1.0, 
                                    alphas=[np.mean(fold_alphas)],
                                    fit_baseline_model=True)
cox_final.fit(X, y)

# Save model and scaler
with open('models/cox_lasso_dysreg.pkl', 'wb') as f:
    pickle.dump(cox_final, f)

with open('models/scaler_dysreg.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary — convert numpy types to plain Python
results = {
    "model": "Cox-Lasso Dysregulation",
    "n_patients": 478,
    "n_genes_input": 819,
    "cv_cindex_mean": round(float(np.mean(fold_cindex)), 3),
    "cv_cindex_std": round(float(np.std(fold_cindex)), 3),
    "fold_cindices": [round(float(c), 4) for c in fold_cindex],
    "fold_alphas": [round(float(a), 4) for a in fold_alphas],
    "fold_genes_kept": [int(g) for g in fold_genes]
}

with open('outputs/results/cox_lasso_dysreg_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved: models/cox_lasso_dysreg.pkl")
print("Saved: models/scaler_dysreg.pkl")
print("Saved: outputs/results/cox_lasso_dysreg_summary.json")
print(f"\nFinal result: C-index = {results['cv_cindex_mean']} ± {results['cv_cindex_std']}")

Saved: models/cox_lasso_dysreg.pkl
Saved: models/scaler_dysreg.pkl
Saved: outputs/results/cox_lasso_dysreg_summary.json

Final result: C-index = 0.559 ± 0.057
